# **Installation**

In [3]:
pip install pandas

In [4]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [10]:
!pip install duckdb duckdb-engine jupysql --quiet

import duckdb
%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [33]:
#Once csv loaded in Collab - run this cell

# 1. Connexion à une base en mémoire
conn = duckdb.connect(database=':memory:')

# 2. Création de table SQL à partir des CSV
conn.execute("CREATE TABLE dropout AS SELECT * FROM read_csv_auto('clinical_trial_dataset_1000(1).csv')")

# 3. Configurer JupySQL
%reload_ext sql
%config SqlMagic.autopandas = True
%sql conn

In [5]:
# 1. CRÉER BASE SQLITE directement dans Colab
import sqlite3
import pandas as pd

# Ta requête SQL → directement DataFrame
query = """
SELECT
  Gender, Age, Health_Condition, Trial_Phase, previous_adherence_score, Dropout_Flag
FROM clinical_dropout.clinical_trial_dataset_1000(1)
"""

# Si tu as accès BigQuery
print("Méthode 1 : SQLite en mémoire - Prêt !")

Méthode 1 : SQLite en mémoire - Prêt !


In [1]:
!pip install pandas scikit-learn joblib

##upload CSV file

In [2]:
from google.colab import files
uploaded = files.upload()

Saving clinical_trial_dataset_1000(1).csv to clinical_trial_dataset_1000(1) (1).csv


In [8]:
import pandas as pd

df = pd.read_csv("clinical_trial_dataset_1000(1).csv")
print("Dataset loaded")
df.head()

Dataset loaded


,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,NaN,Arthritis,0.85,Phase I,No,NaN
1,P0002,32,Male,White,NaN,NaN,0.72,Phase I,No,NaN
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,Yes,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,No,NaN
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,Yes,Medical Advice


# **SQL Coding**

In [34]:
%%sql
SELECT * FROM dropout

Running query in 'DuckDBPyConnection'

,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,None,Arthritis,0.85,Phase I,False,None
1,P0002,32,Male,White,None,None,0.72,Phase I,False,None
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,True,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,False,None
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,True,Medical Advice
...,...,...,...,...,...,...,...,...,...,...
995,P0996,44,Male,Asian,None,None,0.72,Phase I,True,Medical Advice
996,P0997,54,Female,Asian,Hypertension,Arthritis,0.76,Phase II,True,Logistical Issues
997,P0998,61,Male,Hispanic,None,Kidney Disease,0.98,Phase I,False,None
998,P0999,42,Male,Asian,Heart Disease,Depression,0.56,Phase I,True,None


In [44]:
##Total Global participants dropout##

%%sql
SELECT DISTINCT
COUNT(*) AS Total_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout_participants, -- dropout = 1
ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2) As Dropout_rate
FROM dropout

Running query in 'DuckDBPyConnection'

,Total_participants,nb_dropout_participants,Dropout_rate
0,1000,262.0,26.2


In [47]:
%%sql
WITH ranked_dropout AS (
  SELECT
    Gender,
    Health_Condition,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS Dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS risk_rank
  FROM dropout
  GROUP BY Gender, Health_Condition
  HAVING COUNT(*) >= 5 -- better result from 5 participants
)
SELECT
  risk_rank,
  Gender,
  Health_Condition,
  nb_participants,
  nb_dropout,
  Dropout_rate
FROM ranked_dropout
ORDER BY risk_rank

Running query in 'DuckDBPyConnection'

,risk_rank,Gender,Health_Condition,nb_participants,nb_dropout,Dropout_rate
0,1,Male,Asthma,78,25.0,32.05
1,2,Female,None,99,29.0,29.29
2,3,Female,Cancer,110,32.0,29.09
3,4,Female,Hypertension,109,30.0,27.52
4,5,Male,None,62,17.0,27.42
5,6,Female,Asthma,112,30.0,26.79
6,7,Male,Heart Disease,63,16.0,25.40
7,7,Male,Cancer,63,16.0,25.40
8,8,Female,Heart Disease,91,22.0,24.18
9,9,Male,Diabetes,58,14.0,24.14
